In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Flip)=
# Flip

*Flipping a molecular system over a specified plane or normal vector.*

The function {func}`molsysmt.structure.flip` reflects the coordinates of a molecular system over a plane defined by an anchor point and a normal vector.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.structure.flip`.
:::


## Basic usage

Let's show how to flip a POPC lipid molecule across a plane passing through the origin. We first orient its longest principal axis along the vertical $y$-axis using {func}`molsysmt.structure.align_principal_axes` and center it on its phosphorus atom:


In [2]:
import molsysmt as msm
import numpy as np


In [3]:
crd = msm.systems['POPC']['popc.crd']
psf = msm.systems['POPC']['popc.psf']
molsys = msm.convert([crd, psf], to_form='molsysmt.MolSys')
molsys = msm.structure.align_principal_axes(molsys, axes=[[0, 1, 0], [0, 0, 1], [1, 0, 0]])
molsys = msm.structure.center(molsys, selection='all', source_center_selection='atom_name=="P"')


:::{note} Demo Systems Catalog
:class: dropdown

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We inspect the initial 3D orientation of the lipid molecule (with the headgroup pointing upwards along $+y$):


In [4]:
molsysviewer_htmlfile = '_static/views/tools_structure_flip_1.html'


In [5]:
msm.view(molsys)


'<iframe src="../../../../_static/views/tools_structure_flip_1.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

We now reflect the coordinates across the horizontal $xz$-plane (normal vector `[0, 1, 0]`, inverting the vertical $y$-axis) passing through the origin:


In [6]:
molsys_flipped = msm.structure.flip(molsys, vector=[0, 1, 0], point='[0, 0, 0] nm')


Visualizing the flipped system confirms that the lipid is inverted vertically ($y \to -y$):


In [7]:
molsysviewer_htmlfile = '_static/views/tools_structure_flip_2.html'


In [8]:
msm.view(molsys_flipped)


'<iframe src="../../../../_static/views/tools_structure_flip_2.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Default and scaled normals

Omitting `vector` uses [0, 0, 1], reflecting z across the xy plane. Every finite
nonzero scale or sign of a normal defines the same plane; native validation
normalizes it before calling the Rust reflection kernel. Reflection preserves
internal distances and reverses handedness. Stored chemical assignments remain
unchanged.

In [9]:
puw = msm.pyunitwizard
before = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
molsys_default = msm.structure.flip(molsys)
default_xyz = puw.get_value(msm.get(molsys_default, coordinates=True), to_unit='nm')
np.testing.assert_allclose(default_xyz[..., :2], before[..., :2], atol=1e-12)
np.testing.assert_allclose(default_xyz[..., 2], -before[..., 2], atol=1e-12)
molsys_scaled = msm.structure.flip(molsys, vector=[0, -5, 0])
np.testing.assert_allclose(
    puw.get_value(msm.get(molsys_scaled, coordinates=True), to_unit='nm'),
    puw.get_value(msm.get(molsys_flipped, coordinates=True), to_unit='nm'), atol=1e-12)
molsys_restored = msm.structure.flip(molsys_scaled, vector=[0, 2, 0])
np.testing.assert_allclose(
    puw.get_value(msm.get(molsys_restored, coordinates=True), to_unit='nm'), before, atol=1e-12)
print('Default z reflection, normal-scale invariance and double reflection verified.')

Default z reflection, normal-scale invariance and double reflection verified.


## Selecting atoms and structures

`selection` chooses the reflected atoms; other atoms stay fixed. Nonconsecutive
or repeated `structure_indices` retain their original axes. One finite plane
point applies to all requested structures. A point like `[1, 0, 0] nm` may also
be supplied as `[10, 0, 0] angstrom`. Plane normals are unit-free.
`in_place=False` returns an independent system; `in_place=True` returns None.
Native coordinate edits invalidate interaction coverage only for edited structures.
No periodic reconstruction or box transformation is performed.

In [10]:
molsys_ensemble = msm.convert(
    msm.systems['pentalanine']['traj_pentalanine.h5msm'], structure_indices=[0, 8, 3])
original = puw.get_value(msm.get(molsys_ensemble, coordinates=True), to_unit='nm').copy()
molsys_subset = msm.structure.flip(
    molsys_ensemble, vector=[2, 0, 0], point='[10, 0, 0] angstrom',
    selection=[0, 2], structure_indices=[2, 0, 2])
expected = original.copy()
for index in [2, 0]:
    expected[index, [0, 2], 0] = 2 - original[index, [0, 2], 0]
np.testing.assert_allclose(
    puw.get_value(msm.get(molsys_subset, coordinates=True), to_unit='nm'), expected, atol=1e-12)
np.testing.assert_array_equal(
    puw.get_value(msm.get(molsys_ensemble, coordinates=True), to_unit='nm'), original)
print('Selected x=1 nm reflection preserves all other coordinates.')

Selected x=1 nm reflection preserves all other coordinates.


## Rejecting invalid planes before editing

Zero or nonfinite normals do not define a plane. A plane point must be one
finite length-bearing point; a list of several points is rejected rather than
silently using only its first row. Invalid planes reject before writes,
preserving coordinates, boxes and stored interaction coverage.

In [11]:
for options in [{'vector': [0, 0, 0]}, {'vector': [np.nan, 0, 1]},
                {'point': '[[0, 0, 0], [1, 0, 0]] nm'}]:
    try:
        msm.structure.flip(molsys_ensemble, in_place=True, **options)
    except msm.ArgumentError as error:
        print('Rejected without editing:', type(error).__name__)
    else:
        raise AssertionError('An invalid plane unexpectedly succeeded')
    np.testing.assert_array_equal(
        puw.get_value(msm.get(molsys_ensemble, coordinates=True), to_unit='nm'), original)

Rejected without editing: ArgumentError
Rejected without editing: ArgumentError
Rejected without editing: ArgumentError


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Align principal axes <Tutorial_Align_principal_axes>`: Orient principal inertia axes along target Cartesian directions with {func}`molsysmt.structure.align_principal_axes`.
- {ref}`Center <Tutorial_Center>`: Center a molecular system around coordinate origin or reference points with {func}`molsysmt.structure.center`.
- {ref}`View <Tutorial_View>`: Render molecular systems interactively with {func}`molsysmt.basic.view`.
- {ref}`Rotate <Tutorial_Rotate>`: Rotate a molecular system around a chosen axis with {func}`molsysmt.structure.rotate`.
- {ref}`Translate <Tutorial_Translate>`: Translate molecular coordinates by a displacement vector with {func}`molsysmt.structure.translate`.
:::
